# Edge3D Flow-Matching Debug
一个最小可执行的调试 notebook，用来验证单样本 overfit 训练流程已经切到新的 modality-native 语义。
本 notebook 只做三类检查：DataLoader、JiT model forward、LightningModule training step。

In [ ]:
from pathlib import Path
import json
import math

import torch
from utils import load_yaml_config

repo_root = Path.cwd()
config_path = repo_root / "configs/experiment/edge3d_flow_overfit.yaml"
manifest_path = Path("/home/devdata/edge3d_data/manifest_cache/edge3d_overfit.jsonl")
# sample_path = repo_root / "data/samples/edge3d_overfit_0000.npz"
sample_path = Path('/home/devdata/edge3d_data/equirectangular_data/0001c4ddd54a4a9a8afe5e03ed0bd082.npz')

config = load_yaml_config(str(config_path))
manifest_record = json.loads(manifest_path.read_text(encoding="utf-8").splitlines()[0])

print({
    "config": str(config_path),
    "manifest": str(manifest_path),
    "sample": str(sample_path),
    "sample_exists": sample_path.exists(),
    "manifest_record": manifest_record,
})
assert sample_path.exists()

{'config': '/home/viplab/jiaxin/object_panorama/configs/experiment/exp_b32_sparse_debug.yaml', 'manifest': '/home/viplab/jiaxin/object_panorama/data/train_manifest.jsonl', 'sample': '/home/devdata/edge3d_data/equirectangular_data/0001c4ddd54a4a9a8afe5e03ed0bd082.npz', 'sample_exists': True, 'manifest_record': {'sample_id': 'edge3d_overfit_0000', 'tensor_path': '/home/devdata/edge3d_data/equirectangular_data/0001c4ddd54a4a9a8afe5e03ed0bd082.npz', 'meta': {'split': 'train', 'source_uid': '0001c4ddd54a4a9a8afe5e03ed0bd082', 'source_export': 'analysis/edge3d_canonical_edge_100/training_tensors_gpu'}}}


## 1. 准备单样本 Overfit 调试数据
这里直接使用已经重建好的单样本 `.npz`，不再使用旧的 `input/condition/target` 三元组文件。

In [ ]:
sample_meta = {
    "sample_id": manifest_record["sample_id"],
    "sample_path": str(sample_path),
    "sample_size_mb": round(sample_path.stat().st_size / 1024 / 1024, 3),
    "source_folder": manifest_record["meta"]["source_folder"],
}
print(sample_meta)

## 2. 实现最小化 DataLoader
DataLoader 只返回 `{model_rgb, model_depth, model_normal, edge_depth}`，不在数据层构造 `input/condition/target`。

In [ ]:
from datasets import build_dataset_from_config, build_dataloader_from_config

dataset = build_dataset_from_config(config["data"]["train"])
loader = build_dataloader_from_config(
    {"batch_size": 1, "num_workers": 0, "shuffle": False},
    dataset,
    max_condition_channels=35,
)
batch = next(iter(loader))

batch_shapes = {key: list(value.shape) for key, value in batch.items() if torch.is_tensor(value)}
print("dataset_len:", len(dataset))
print("batch_keys:", list(batch.keys()))
print("batch_shapes:", batch_shapes)

assert set(batch.keys()) == {"sample_ids", "model_rgb", "model_depth", "model_normal", "edge_depth", "meta"}
assert batch["edge_depth"].shape[1] == 3

## 3. 验证多模态 Condition 通道拼接
单个 hit 的模态通道数是 `3 + 1 + 3 = 7`，当前导出保留 `5` 个 model hit，所以实际总 condition 通道数是 `35`。

In [ ]:
condition = torch.cat([batch["model_rgb"], batch["model_depth"], batch["model_normal"]], dim=1)
per_hit_channels = 3 + 1 + 3
model_hits = batch["model_depth"].shape[1]
total_condition_channels = condition.shape[1]

print({
    "per_hit_channels": per_hit_channels,
    "model_hits": model_hits,
    "total_condition_channels": total_condition_channels,
})

assert per_hit_channels == 7
assert model_hits == 5
assert total_condition_channels == 35
assert condition.shape[-2:] == batch["edge_depth"].shape[-2:]

## 4. 构建最小 JiT Flow-Matching 模型
这里直接复用仓库里的 `RectangularConditionalJiTModel` 配置，只验证最基本的前向路径。

In [ ]:
from models import create_rectangular_conditional_jit_model

model_cfg = dict(config["model"])
model_cfg.pop("name", None)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = create_rectangular_conditional_jit_model(**model_cfg).to(device)
model.eval()

edge_target = torch.nan_to_num(batch["edge_depth"].to(device), nan=0.0, posinf=0.0, neginf=0.0)
synthetic_condition = torch.nan_to_num(condition.to(device), nan=0.0, posinf=0.0, neginf=0.0)
t = torch.full((edge_target.shape[0],), 0.5, device=device, dtype=edge_target.dtype)

with torch.no_grad():
    forward_out = model(
        sample=edge_target,
        timestep=t,
        condition=synthetic_condition,
        condition_type_ids=torch.zeros(edge_target.shape[0], device=device, dtype=torch.long),
    )

print({
    "device": str(device),
    "forward_shape": list(forward_out.sample.shape),
    "is_finite": bool(torch.isfinite(forward_out.sample).all().item()),
})
assert tuple(forward_out.sample.shape) == tuple(edge_target.shape)

## 5. 实现精简版 RectangularConditionalJiTLightningModule
训练模块只保留 flow-matching 语义：`target=edge_depth`，`condition=concat(model_rgb, model_depth, model_normal)`，`sample` 为加噪后的 target。

In [ ]:
from training.lightning_module import RectangularConditionalJiTLightningModule, _build_model_input_batch

module = RectangularConditionalJiTLightningModule(
    model_cfg=dict(config["model"]),
    objective_cfg=dict(config["objective"]),
    loss_cfg=dict(config["loss"]),
    optim_cfg=dict(config["train"]),
    freeze_cfg=dict(config.get("freeze", {})),
    pretrained_cfg=dict(config.get("pretrained", {})),
).to(device)
module.eval()

runtime_batch = {
    "sample_ids": batch["sample_ids"],
    "model_rgb": batch["model_rgb"].to(device),
    "model_depth": batch["model_depth"].to(device),
    "model_normal": batch["model_normal"].to(device),
    "edge_depth": batch["edge_depth"].to(device),
    "meta": batch["meta"],
}
model_input = _build_model_input_batch(runtime_batch, dict(config["objective"]))

print({
    "sample_shape": list(model_input.sample.shape),
    "condition_shape": list(model_input.condition.shape),
    "target_shape": list(model_input.target.shape),
    "t_min": float(model_input.timestep.min().item()),
    "t_max": float(model_input.timestep.max().item()),
})
assert model_input.sample.shape == model_input.target.shape
assert model_input.condition.shape[1] == 35

## 6. 执行单步前向与损失检查
这里同时检查 noisy input 构造、模型输出 shape、loss 有限性。

In [ ]:
module.train()
loss = module.training_step(runtime_batch, batch_idx=0)

with torch.no_grad():
    train_out = module.model(
        sample=model_input.sample,
        timestep=model_input.timestep,
        condition=model_input.condition,
        condition_type_ids=model_input.condition_type_ids,
    )

print({
    "loss": float(loss.detach().item()),
    "loss_is_finite": bool(torch.isfinite(loss).item()),
    "pred_shape": list(train_out.sample.shape),
    "target_shape": list(model_input.target.shape),
    "sample_is_noisy_target": bool(not torch.allclose(model_input.sample, model_input.target)),
})
assert torch.isfinite(loss)
assert train_out.sample.shape == model_input.target.shape

## 7. 执行单样本 Overfit 训练循环
只跑少量 step，确认整条链路可反向传播并保持数值稳定。

In [ ]:
from tqdm import tqdm
module.train()
optimizer = torch.optim.AdamW(module.model.parameters(), lr=1.0e-5)
loss_history = []

for step in tqdm(range(300)):
    optimizer.zero_grad(set_to_none=True)
    step_loss = module.training_step(runtime_batch, batch_idx=step)
    step_loss.backward()
    optimizer.step()
    loss_history.append(float(step_loss.detach().cpu().item()))

print({"loss_history": loss_history})
assert all(math.isfinite(value) for value in loss_history)

## 8. 可视化 Input / Condition / Target 与预测结果
这里把第一层的可视化放在一起，方便快速检查数据语义和预测输出是否对齐。

In [ ]:
import matplotlib.pyplot as plt

module.eval()
with torch.no_grad():
    refreshed_model_input = _build_model_input_batch(runtime_batch, dict(config["objective"]))
    pred = module.model(
        sample=refreshed_model_input.sample,
        timestep=refreshed_model_input.timestep,
        condition=refreshed_model_input.condition,
        condition_type_ids=refreshed_model_input.condition_type_ids,
    ).sample

hit_index = 0

rgb_hit = batch["model_rgb"][0, 3*hit_index:3*(hit_index+1)].permute(1, 2, 0).detach().cpu()
rgb_hit = (rgb_hit - rgb_hit.min()) / (rgb_hit.max() - rgb_hit.min() + 1e-8)
normal_hit = batch["model_normal"][0, 3*hit_index:3*(hit_index+1)].permute(1, 2, 0).detach().cpu()
normal_hit = (normal_hit - normal_hit.min()) / (normal_hit.max() - normal_hit.min() + 1e-8)
edge_target_hit = torch.nan_to_num(batch["edge_depth"][0, hit_index].detach().cpu(), nan=0.0, posinf=0.0, neginf=0.0)

vis = {
    f"model_rgb_hit{hit_index}": rgb_hit,
    f"model_depth_hit{hit_index}": batch["model_depth"][0, hit_index].detach().cpu(),
    f"model_normal_hit{hit_index}": normal_hit,
    f"edge_target_hit{hit_index}": edge_target_hit,
    f"noisy_input_hit{hit_index}": refreshed_model_input.sample[0, hit_index].detach().cpu(),
    f"pred_hit{hit_index}": pred[0, hit_index].detach().cpu(),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (title, image) in zip(axes.flatten(), vis.items()):
    ax.set_title(title)
    if image.ndim == 3:
        ax.imshow(image.numpy())
    else:
        ax.imshow(image.numpy(), cmap="viridis")
    ax.axis("off")
plt.tight_layout()
plt.show()

summary = {
    name: {"min": float(image.min().item()), "max": float(image.max().item())}
    for name, image in vis.items()
    if image.ndim == 2
}
print(summary)